In [2]:
import rbfopt
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="Signature .* for <class 'numpy.longdouble'> does not match any known type")
import csv

In [4]:
def load_grid(filename="alpine2_30x30.csv"):
    """
    Load grid data from CSV.
    Expected CSV columns: "x", "y", "Objective", among others.
    Returns a dictionary mapping (x, y) tuples to the objective value.
    """
    grid_data = {}
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                # Convert x and y values to integers
                x_val = int(row["x"])
                y_val = int(row["y"])
                # Convert objective value to float
                obj_val = float(row["Objective"])
                grid_data[(x_val, y_val)] = obj_val
            except Exception as e:
                # Skip rows with conversion issues
                continue
    return grid_data
# Load the data from grid.csv
grid_data = load_grid()

# Determine grid bounds from the keys of grid_data
xs = [pt[0] for pt in grid_data.keys()]
ys = [pt[1] for pt in grid_data.keys()]
min_x, max_x = min(xs), max(xs)
min_y, max_y = min(ys), max(ys)
print("Grid bounds:")
print("x in [{}, {}]".format(min_x, max_x))
print("y in [{}, {}]".format(min_y, max_y))

Grid bounds:
x in [0, 30]
y in [0, 30]


In [5]:
# Load the data from grid.csv
grid_data = load_grid()

# PENALTY = 1e5  # adjust this penalty as appropriate
PENALTY = np.inf # use infinity as penalty for infeasible points
# PENALTY = np.nan  # use NaN as penalty for infeasible points
# PENALTY = None  # use None as penalty for infeasible points

def obj_funct(x):
    """
    Given a candidate point x (a list or 1-D numpy array with 2 elements),
    this function rounds x to the nearest integers and returns the objective value
    if the point exists in grid_data and is feasible (i.e. y <= x).
    Otherwise, returns a penalty value.
    """
    # Round the candidate coordinates to nearest integers
    x_int = int(round(x[0]))
    y_int = int(round(x[1]))
        
    # If the point exists in the grid, return its objective value.
    if (x_int, y_int) in grid_data:
        return grid_data[(x_int, y_int)]
    else:
        # If the point is not in the grid, return the penalty.
        return PENALTY

In [7]:
test_point = [27,27]
value = obj_funct(test_point)
print(f"For point {test_point} the objective value is {value}")

For point [27, 27] the objective value is -896.0502370024856


In [12]:
# Set up the black-box for a 2D problem with integer variables.
# We use the determined bounds and mark both variables as integer ("I").
black_box = rbfopt.RbfoptUserBlackBox(
    2,                   # number of variables
    np.array([0, 0]),    # lower bounds for x and y
    np.array([30, 30]),    # upper bounds for x and y
    np.array(["I", "I"]),# both variables are integers
    obj_funct
)

# Configure optimization settings.
settings = rbfopt.RbfoptSettings(
    max_evaluations=90,  # maximum number of evaluations
    minlp_solver_path='/home/albertlee/ampl.linux-intel64/bonmin',
    nlp_solver_path='/home/albertlee/ampl.linux-intel64/ipopt',
    algorithm='MSRSM',   # optimization algorithm variant
    rbf='auto',           # allow automatic RBF selection
    global_search_method = 'solver',  # options = ['sampling', 'genetic', 'solver']
)

# Create the optimization algorithm instance.
alg = rbfopt.RbfoptAlgorithm(settings, black_box)

# Run the optimization.
best_val, best_point, itercount, evalcount, fast_evalcount = alg.optimize()

print("Optimization completed.")
print("Best objective value found:", best_val)
print("At lattice point (rounded):", [int(round(v)) for v in best_point])
print("Iterations:", itercount, "Function evaluations:", evalcount)

  Iter  Cycle  Action             Objective value      Time      Gap
  ----  -----  ------             ---------------      ----      ---
     0      0  Initialization         -782.723595      0.01   100.00 *
     0      0  Initialization            0.000000      0.01   100.00  
     0      0  Initialization         -109.405231      0.01   100.00  
     0      0  GlobalStep               -0.000000      0.22   100.00  
     1      0  GlobalStep               -0.000000      0.45   100.00  
     2      0  GlobalStep             -683.729775      0.83   100.00  
     3      0  GlobalStep              547.382604      1.09   100.00  
     4      0  GlobalStep              221.006360      1.37   100.00  
     5      0  AdjLocalStep            547.382604      1.83   100.00  
     6      1  GlobalStep               -0.000000      2.19   100.00  
     7      1  GlobalStep                5.480115      2.50   100.00  
     8      1  GlobalStep              101.914846      2.89   100.00  
     9    